
<a id='np'></a>
<div id="qe-notebook-header" align="right" style="text-align:right;">
        <a href="https://quantecon.org/" title="quantecon.org">
                <img style="width:250px;display:inline;" width="250px" src="https://assets.quantecon.org/img/qe-menubar-logo.svg" alt="QuantEcon">
        </a>
</div>

# NumPy


<a id='index-1'></a>

## Contents

- [NumPy](#NumPy)  
  - [Overview](#Overview)  
  - [NumPy Arrays](#NumPy-Arrays)  
  - [Operations on Arrays](#Operations-on-Arrays)  
  - [Additional Functionality](#Additional-Functionality)  
  - [Exercises](#Exercises)  
  - [Solutions](#Solutions)  

> “Let’s be clear: the work of science has nothing whatever to do with consensus.  Consensus is the business of politics. Science, on the contrary, requires only one investigator who happens to be right, which means that he or she has results that are verifiable by reference to the real world. In science consensus is irrelevant. What is relevant is reproducible results.” – Michael Crichton

## Overview

[NumPy](https://en.wikipedia.org/wiki/NumPy) is a first-rate library for numerical programming

- Widely used in academia, finance and industry.  
- Mature, fast, stable and under continuous development.  


We have already seen some code involving NumPy in the preceding lectures.

In this lecture, we will start a more systematic discussion of both

- NumPy arrays and  
- the fundamental array processing operations provided by NumPy.  

### References

- [The official NumPy documentation](http://docs.scipy.org/doc/numpy/reference/).  



<a id='numpy-array'></a>

## NumPy Arrays


<a id='index-2'></a>
The essential problem that NumPy solves is fast array processing.

The most important structure that NumPy defines is an array data type formally called a [numpy.ndarray](http://docs.scipy.org/doc/numpy/reference/arrays.ndarray.html).

NumPy arrays power a large proportion of the scientific Python ecosystem.

Let’s first import the library.

In [1]:
import numpy as np

To create a NumPy array containing only zeros we use  [np.zeros](http://docs.scipy.org/doc/numpy/reference/generated/numpy.zeros.html#numpy.zeros)

In [17]:
import numpy as np
import math

def generate_cmry_instance(nq=12, n_lin=None, n_nonlin=6):
    if n_lin is None:
        n_lin = math.ceil(math.log2(nq))

    if nq % 3 != 0:
        raise ValueError("nq must be a multiple of 3 for 3-qubit gate topology.")

    gates_per_layer = nq // 3

    # --- 1. Parameter Sampling ---

    def sample_3x3_invertible():
        """Samples a random 3x3 invertible matrix over GF(2)."""
        while True:
            M = np.random.randint(0, 2, (3, 3))
            if round(np.linalg.det(M)) % 2 != 0:
                return M

    import numpy as np

    def sample_inflationary_3x3():
        """
        Samples one of the 18 strictly inflationary 3x3 invertible matrices over GF(2).
        Every column must have a Hamming weight >= 2.
        """
        valid_columns = np.array([
            [1, 1, 0],
            [1, 0, 1],
            [0, 1, 1],
            [1, 1, 1]
        ])

        while True:
            # Select 3 distinct columns from the 4 available
            idx = np.random.choice(4, 3, replace=False)
            M = valid_columns[idx].T

            # Ensure the selected columns are linearly independent over GF(2)
            if round(np.linalg.det(M)) % 2 != 0:
                return M

# The shift vectors 'c' are still sampled uniformly from the 8 possibilities
# Together, M and c strictly map to one of the 144 physical gates.

    # Parameters for L_x
    # Lx_M = [[sample_3x3_invertible() for _ in range(gates_per_layer)] for _ in range(n_lin)]
    Lx_M = [[sample_inflationary_3x3() for _ in range(gates_per_layer)] for _ in range(n_lin)]
    Lx_c = [[np.random.randint(0, 2, 3) for _ in range(gates_per_layer)] for _ in range(n_lin)]

    # Parameters for L_beta
    # Lb_M = [[sample_3x3_invertible() for _ in range(gates_per_layer)] for _ in range(n_lin)]
    Lb_M = [[sample_inflationary_3x3() for _ in range(gates_per_layer)] for _ in range(n_lin)]

    Lb_c = [[np.random.randint(0, 2, 3) for _ in range(gates_per_layer)] for _ in range(n_lin)]

    # Parameters for N (Permutations of integers 0-7)
    N_perms = [[np.random.permutation(8) for _ in range(gates_per_layer)] for _ in range(n_nonlin)]

    # Key beta
    beta = np.random.randint(0, 2, nq)
    # Ensure beta is non-zero
    if np.sum(beta) == 0:
        beta[0] = 1

    # --- 2. Forward Pass Mechanics ---

    def apply_routing(state):
        """Simulates global mixing between layers to mimic ternary tree stride."""
        # Reshape to (3, nq/3), transpose, and flatten to interleave bits
        return state.reshape((3, gates_per_layer), order='F').flatten()

    def apply_affine_block(x, M_params, c_params):
        """Applies a full affine block (L_x or L_beta) over GF(2)."""
        state = x.copy()
        for layer in range(len(M_params)):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                idx = slice(g * 3, g * 3 + 3)
                state[idx] = (M_params[layer][g] @ state[idx] + c_params[layer][g]) % 2
        return state

    def apply_nonlinear_block(x):
        """Applies the core non-linear permutations."""
        state = x.copy()
        for layer in range(n_nonlin):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                idx = slice(g * 3, g * 3 + 3)
                # Convert 3-bit binary array to integer (LSB at index 0)
                val = state[idx.start] + 2*state[idx.start+1] + 4*state[idx.start+2]
                out_val = N_perms[layer][g][val]
                # Convert integer back to 3-bit array
                state[idx.start]   = out_val & 1
                state[idx.start+1] = (out_val >> 1) & 1
                state[idx.start+2] = (out_val >> 2) & 1
        return state

    # --- 3. Global Matrix Extraction ---

    zero_state = np.zeros(nq, dtype=int)

    # Extract d and C from L_x
    d = apply_affine_block(zero_state, Lx_M, Lx_c)
    C = np.zeros((nq, nq), dtype=int)
    for i in range(nq):
        e_i = np.zeros(nq, dtype=int)
        e_i[i] = 1
        C[:, i] = apply_affine_block(e_i, Lx_M, Lx_c) ^ d

    # Extract b_vec and A from L_beta
    b_vec = apply_affine_block(zero_state, Lb_M, Lb_c)
    A = np.zeros((nq, nq), dtype=int)
    for i in range(nq):
        e_i = np.zeros(nq, dtype=int)
        e_i[i] = 1
        A[:, i] = apply_affine_block(e_i, Lb_M, Lb_c) ^ b_vec

    # Collapse the outer bookend into the effective mask
    beta_prime = (A.T @ beta) % 2

    # --- 4. Truth Table Evaluation ---

    num_states = 2**nq
    F = np.zeros(num_states, dtype=int)
    H_matrix = np.zeros((num_states, nq), dtype=int) # Stores h(x) for all x

    # Pre-calculate the global phase constant
    phase_shift = np.dot(beta, b_vec) % 2

    for i in range(num_states):
        # Generate input bitstring (Standard order: bit 0 is LSB)
        x = np.array([(i >> j) & 1 for j in range(nq)], dtype=int)

        # 1. z = Cx ^ d
        z = (C @ x + d) % 2

        # 2. h = N(z)
        h = apply_nonlinear_block(z)
        H_matrix[i, :] = h

        # 3. f(x) = (beta' dot h) ^ (beta dot b_vec)
        F[i] = (np.dot(beta_prime, h) + phase_shift) % 2

    return F, beta, C, d, A, H_matrix, b_vec, beta_prime

# Example Execution
F, beta, C, d, A, H_matrix, b_vec, beta_prime = generate_cmry_instance(nq=12)

In [25]:
import numpy as np
import itertools
import math

# --- 1. Dictionary Precomputation ---

def get_inflationary_matrices():
    """Returns the 18 strictly inflationary 3x3 invertible matrices over GF(2)."""
    valid_columns = np.array([
        [1, 1, 0],
        [1, 0, 1],
        [0, 1, 1],
        [1, 1, 1]
    ])
    matrices = []
    # Test all combinations of 3 columns
    for idx in itertools.permutations(range(4), 3):
        M = valid_columns[list(idx)].T
        if round(np.linalg.det(M)) % 2 != 0:  # Must be invertible
            matrices.append(M)
    return matrices

def get_super_nonlinear_permutations():
    """
    Filters S_8 to return the 10,752 super-nonlinear 3-bit permutations.
    A permutation is super-nonlinear if the max absolute Walsh coefficient
    of all 7 non-zero linear combinations of its outputs is exactly 4.
    """
    valid_perms = []

    # Fast bit-count for dot products over GF(2)
    def dot_gf2(a, b):
        return bin(a & b).count('1') % 2

    for perm in itertools.permutations(range(8)):
        is_super_nl = True
        # Check all 7 non-zero output linear combinations
        for v in range(1, 8):
            f = [dot_gf2(v, p) for p in perm]

            # Compute Walsh-Hadamard Transform of f
            max_walsh = 0
            for u in range(8):
                walsh_val = sum((-1)**(f[x] ^ dot_gf2(u, x)) for x in range(8))
                if abs(walsh_val) > max_walsh:
                    max_walsh = abs(walsh_val)

            # Max possible nonlinearity for n=3 means max Walsh magnitude is 4
            if max_walsh > 4:
                is_super_nl = False
                break

        if is_super_nl:
            valid_perms.append(np.array(perm))

    return valid_perms

# Precompute the dictionaries once
DICT_INFLATIONARY_MATRICES = get_inflationary_matrices()
DICT_SUPER_NONLINEAR_PERMS = get_super_nonlinear_permutations()


# --- 2. CMRY Instance Generation ---

def generate_strict_cmry_instance(nq=12, n_lin=None, n_nonlin=6):
    if n_lin is None:
        n_lin = math.ceil(math.log2(nq))

    if nq % 3 != 0:
        raise ValueError("nq must be a multiple of 3 for 3-qubit gate topology.")

    gates_per_layer = nq // 3

    # Sample L_x parameters from the 18 valid matrices and 8 valid shifts (144 total gates)
    Lx_M = [[DICT_INFLATIONARY_MATRICES[np.random.choice(18)] for _ in range(gates_per_layer)] for _ in range(n_lin)]
    Lx_c = [[np.random.randint(0, 2, 3) for _ in range(gates_per_layer)] for _ in range(n_lin)]

    # Sample L_beta parameters from the 18 valid matrices and 8 valid shifts
    Lb_M = [[DICT_INFLATIONARY_MATRICES[np.random.choice(18)] for _ in range(gates_per_layer)] for _ in range(n_lin)]
    Lb_c = [[np.random.randint(0, 2, 3) for _ in range(gates_per_layer)] for _ in range(n_lin)]

    # Sample N parameters from the 10,752 super-nonlinear permutations
    N_perms = [[DICT_SUPER_NONLINEAR_PERMS[np.random.choice(10752)] for _ in range(gates_per_layer)] for _ in range(n_nonlin)]

    # Key beta
    beta = np.random.randint(0, 2, nq)
    if np.sum(beta) == 0:
        beta[0] = 1

    # --- Forward Pass Mechanics ---

    def apply_routing(state):
        return state.reshape((3, gates_per_layer), order='F').flatten()

    def apply_affine_block(x, M_params, c_params):
        state = x.copy()
        for layer in range(len(M_params)):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                idx = slice(g * 3, g * 3 + 3)
                state[idx] = (M_params[layer][g] @ state[idx] + c_params[layer][g]) % 2
        return state

    def apply_nonlinear_block(x):
        state = x.copy()
        for layer in range(n_nonlin):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                idx = slice(g * 3, g * 3 + 3)
                val = state[idx.start] + 2*state[idx.start+1] + 4*state[idx.start+2]
                out_val = N_perms[layer][g][val]
                state[idx.start]   = out_val & 1
                state[idx.start+1] = (out_val >> 1) & 1
                state[idx.start+2] = (out_val >> 2) & 1
        return state

    # --- Global Matrix Extraction ---

    zero_state = np.zeros(nq, dtype=int)

    # Extract d and C from L_x
    d = apply_affine_block(zero_state, Lx_M, Lx_c)
    C = np.zeros((nq, nq), dtype=int)
    for i in range(nq):
        e_i = np.zeros(nq, dtype=int)
        e_i[i] = 1
        C[:, i] = apply_affine_block(e_i, Lx_M, Lx_c) ^ d

    # Extract b_vec and A from L_beta
    b_vec = apply_affine_block(zero_state, Lb_M, Lb_c)
    A = np.zeros((nq, nq), dtype=int)
    for i in range(nq):
        e_i = np.zeros(nq, dtype=int)
        e_i[i] = 1
        A[:, i] = apply_affine_block(e_i, Lb_M, Lb_c) ^ b_vec

    beta_prime = (A.T @ beta) % 2

    # --- Truth Table Evaluation ---

    num_states = 2**nq
    F = np.zeros(num_states, dtype=int)
    H_matrix = np.zeros((num_states, nq), dtype=int)
    phase_shift = np.dot(beta, b_vec) % 2

    for i in range(num_states):
        x = np.array([(i >> j) & 1 for j in range(nq)], dtype=int)

        z = (C @ x + d) % 2
        h = apply_nonlinear_block(z)
        H_matrix[i, :] = h

        F[i] = (np.dot(beta_prime, h) + phase_shift) % 2

    return F, beta, C, d, A, H_matrix, b_vec, beta_prime

# Execute strict sampling
F, beta, C, d, A, H_matrix, b_vec, beta_prime = generate_strict_cmry_instance(nq=12)

In [26]:
F

array([1, 0, 0, ..., 1, 0, 1])

In [27]:
len(F)

4096

In [28]:
beta

array([0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1])

In [29]:
C

array([[0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1],
       [0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1],
       [1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0],
       [0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0],
       [1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1],
       [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1],
       [1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0],
       [0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1],
       [0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0]])

In [30]:
d

array([1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0])

In [31]:
A

array([[0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0],
       [1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
       [0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0],
       [0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1],
       [0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0],
       [0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0],
       [0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1],
       [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1],
       [1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0],
       [1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0],
       [1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1],
       [0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1]])

In [32]:
H_matrix

array([[1, 0, 0, ..., 1, 1, 1],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [1, 1, 0, ..., 1, 1, 0],
       [1, 0, 0, ..., 0, 1, 0],
       [0, 0, 1, ..., 1, 0, 1]])

In [33]:
b_vec

array([1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0])

In [34]:
beta_prime

array([0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0])

In [ ]:
a = np.zeros(3)
a

In [40]:
F, beta, C, d, A, H_matrix, b_vec, beta_prime = generate_strict_cmry_instance(nq=18)

In [41]:
F

array([1, 0, 0, ..., 1, 0, 1])

In [39]:
F.shape

(32768,)

In [42]:
import numpy as np
import itertools
import math
import warnings

# --- 1. Dictionary Precomputation ---

def get_144_linear_inflationary_gates():
    """Returns a list of 144 tuples (M, c) defining the strictly inflationary affine gates."""
    valid_columns = np.array([
        [1, 1, 0], [1, 0, 1], [0, 1, 1], [1, 1, 1]
    ])
    matrices = []
    # Find the 18 valid invertible matrices
    for idx in itertools.permutations(range(4), 3):
        M = valid_columns[list(idx)].T
        if round(np.linalg.det(M)) % 2 != 0:
            matrices.append(M)

    # Generate all 8 possible 3-bit shift vectors
    shifts = [np.array([(i >> 0) & 1, (i >> 1) & 1, (i >> 2) & 1]) for i in range(8)]

    # Combine to form the exact 144 affine gates
    gates_144 = [(M, c) for M in matrices for c in shifts]
    return gates_144

def get_10752_super_nonlinear_permutations():
    """Filters S_8 to return the 10,752 super-nonlinear 3-bit permutations."""
    print("[INIT] Building dictionary of 10,752 super-nonlinear permutations. This takes ~2-5 seconds...")
    valid_perms = []

    def dot_gf2(a, b):
        return bin(a & b).count('1') % 2

    for perm in itertools.permutations(range(8)):
        is_super_nl = True
        for v in range(1, 8):
            f = [dot_gf2(v, p) for p in perm]
            max_walsh = 0
            for u in range(8):
                walsh_val = sum((-1)**(f[x] ^ dot_gf2(u, x)) for x in range(8))
                if abs(walsh_val) > max_walsh:
                    max_walsh = abs(walsh_val)
            if max_walsh > 4:
                is_super_nl = False
                break
        if is_super_nl:
            valid_perms.append(np.array(perm))

    print("[INIT] Dictionary construction complete.")
    return valid_perms

# Precompute the dictionaries once globally
DICT_144_LINEAR = get_144_linear_inflationary_gates()
DICT_10752_NONLINEAR = get_10752_super_nonlinear_permutations()


# --- 2. CMRY Instance Generation ---

def generate_strict_cmry_instance(nq=12, n_lin=None, n_nonlin=6):
    if n_lin is None:
        n_lin = math.ceil(math.log2(nq))

    if nq % 3 != 0:
        raise ValueError(f"nq={nq} is invalid. Qubit count must be a multiple of 3 to fit the topology.")

    gates_per_layer = nq // 3

    # --- A. Parameter Sampling (The Ground Truth Unknowns) ---

    # Sample integer IDs indexing the precomputed dictionaries
    Lx_indices = np.random.randint(0, 144, (n_lin, gates_per_layer))
    Lb_indices = np.random.randint(0, 144, (n_lin, gates_per_layer))
    N_indices  = np.random.randint(0, 10752, (n_nonlin, gates_per_layer))

    # Sample the original hidden key
    beta = np.random.randint(0, 2, nq)
    if np.sum(beta) == 0:
        beta[0] = 1

    # --- B. Forward Pass Mechanics ---

    def apply_routing(state):
        return state.reshape((3, gates_per_layer), order='F').flatten()

    def apply_affine_block(x, indices_matrix):
        state = x.copy()
        for layer in range(len(indices_matrix)):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                gate_id = indices_matrix[layer, g]
                M, c = DICT_144_LINEAR[gate_id]
                idx = slice(g * 3, g * 3 + 3)
                state[idx] = (M @ state[idx] + c) % 2
        return state

    def apply_nonlinear_block(x):
        state = x.copy()
        for layer in range(n_nonlin):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                gate_id = N_indices[layer, g]
                perm = DICT_10752_NONLINEAR[gate_id]
                idx = slice(g * 3, g * 3 + 3)

                # integer rep -> array lookup -> array rep
                val = state[idx.start] + 2*state[idx.start+1] + 4*state[idx.start+2]
                out_val = perm[val]
                state[idx.start]   = (out_val >> 0) & 1
                state[idx.start+1] = (out_val >> 1) & 1
                state[idx.start+2] = (out_val >> 2) & 1
        return state

    # --- C. Global Matrix Extraction ---

    zero_state = np.zeros(nq, dtype=int)

    # Extract d and C from L_x
    d = apply_affine_block(zero_state, Lx_indices)
    C = np.zeros((nq, nq), dtype=int)
    for i in range(nq):
        e_i = np.zeros(nq, dtype=int)
        e_i[i] = 1
        C[:, i] = apply_affine_block(e_i, Lx_indices) ^ d

    # Extract b_vec and A from L_beta
    b_vec = apply_affine_block(zero_state, Lb_indices)
    A = np.zeros((nq, nq), dtype=int)
    for i in range(nq):
        e_i = np.zeros(nq, dtype=int)
        e_i[i] = 1
        A[:, i] = apply_affine_block(e_i, Lb_indices) ^ b_vec

    # Calculate the effective collapsed mask (Unknown #3)
    beta_prime = (A.T @ beta) % 2

    # Group the specific parameters the solver must learn (flattened for clean vector input)
    learnable_parameters = (
        Lx_indices.flatten(),
        N_indices.flatten(),
        beta_prime.flatten()
    )

    # --- D. Batched Evaluator ---

    phase_shift = np.dot(beta, b_vec) % 2

    def evaluate_batch(X_batch):
        """
        Evaluates f(x) for a list/array of 1D binary vectors.
        Returns a 1D numpy array of boolean bits.
        """
        F_batch = np.zeros(len(X_batch), dtype=int)
        for i, x in enumerate(X_batch):
            z = (C @ x + d) % 2
            h = apply_nonlinear_block(z)
            F_batch[i] = (np.dot(beta_prime, h) + phase_shift) % 2
        return F_batch

    # --- E. Execution Routing (Safety Check) ---

    F_full = None
    if nq > 18:
        msg = (f"\n[DISCLAIMER] nq={nq} > 18. Generating the full truth table "
               f"(2^{nq} items) is exponentially slow and memory-intensive.\n"
               f"-> Returning F_full as None.\n"
               f"-> Use the returned 'evaluator' function to compute specific f(x) batches.\n")
        warnings.warn(msg)
        print(msg)
    else:
        num_states = 2**nq
        F_full = np.zeros(num_states, dtype=int)
        # We can optimize generating the full truth table by batching it
        X_all = np.array([[(i >> j) & 1 for j in range(nq)] for i in range(num_states)])
        F_full = evaluate_batch(X_all)

    # Return a comprehensive dictionary
    return {
        'learnable_params': learnable_parameters, # (Lx_idx_vec, N_idx_vec, beta_prime_vec)
        'evaluator': evaluate_batch,              # Callable function for sampled dataset
        'F_full': F_full,                         # Full Truth Table (or None)
        'beta': beta,
        'C': C,
        'd': d,
        'A': A,
        'b_vec': b_vec,
        'beta_prime': beta_prime,
        'Lx_indices': Lx_indices,
        'Lb_indices': Lb_indices,
        'N_indices': N_indices
    }

# --- Example Usage ---
print("\n--- Generating Instance for nq=12 ---")
cmry_12 = generate_strict_cmry_instance(nq=12)
unknowns_12 = cmry_12['learnable_params']
print(f"Lx Integer Params: {len(unknowns_12[0])}")
print(f"N  Integer Params: {len(unknowns_12[1])}")
print(f"Beta' Boolean Params: {len(unknowns_12[2])}")
print(f"Total Unknowns: {len(unknowns_12[0]) + len(unknowns_12[1]) + len(unknowns_12[2])}")

print("\n--- Generating Instance for nq=24 ---")
cmry_24 = generate_strict_cmry_instance(nq=24)
# Will trigger the disclaimer and bypass F_full execution.

[INIT] Building dictionary of 10,752 super-nonlinear permutations. This takes ~2-5 seconds...
[INIT] Dictionary construction complete.

--- Generating Instance for nq=12 ---
Lx Integer Params: 16
N  Integer Params: 24
Beta' Boolean Params: 12
Total Unknowns: 52

--- Generating Instance for nq=24 ---

[DISCLAIMER] nq=24 > 18. Generating the full truth table (2^24 items) is exponentially slow and memory-intensive.
-> Returning F_full as None.
-> Use the returned 'evaluator' function to compute specific f(x) batches.



/tmp/ipykernel_3565/955426842.py:168: UserWarning: 
[DISCLAIMER] nq=24 > 18. Generating the full truth table (2^24 items) is exponentially slow and memory-intensive.
-> Returning F_full as None.
-> Use the returned 'evaluator' function to compute specific f(x) batches.

  warnings.warn(msg)


In [1]:
import numpy as np
import itertools
import math
import warnings

# --- 1. Global Dictionary Precomputation ---
# (Included here so the script runs standalone, but normally you'd run this once and save it)

def get_144_linear_inflationary_gates():
    valid_columns = np.array([[1, 1, 0], [1, 0, 1], [0, 1, 1], [1, 1, 1]])
    matrices = []
    for idx in itertools.permutations(range(4), 3):
        M = valid_columns[list(idx)].T
        if round(np.linalg.det(M)) % 2 != 0:
            matrices.append(M)
    shifts = [np.array([(i >> 0) & 1, (i >> 1) & 1, (i >> 2) & 1]) for i in range(8)]
    return [(M, c) for M in matrices for c in shifts]

def get_10752_super_nonlinear_permutations():
    valid_perms = []
    def dot_gf2(a, b): return bin(a & b).count('1') % 2
    for perm in itertools.permutations(range(8)):
        is_super_nl = True
        for v in range(1, 8):
            f = [dot_gf2(v, p) for p in perm]
            max_walsh = max(abs(sum((-1)**(f[x] ^ dot_gf2(u, x)) for x in range(8))) for u in range(8))
            if max_walsh > 4:
                is_super_nl = False
                break
        if is_super_nl:
            valid_perms.append(np.array(perm))
    return valid_perms

# Compute once
DICT_144_LINEAR = get_144_linear_inflationary_gates()
DICT_10752_NONLINEAR = get_10752_super_nonlinear_permutations()

# --- 2. Instance Generator (Updated to return dicts) ---

def generate_strict_cmry_instance(nq=12, n_lin=None, n_nonlin=6):
    """Generates a ground-truth CMRY instance and extracts learnable parameters."""
    if n_lin is None:
        n_lin = math.ceil(math.log2(nq))

    assert nq % 3 == 0, f"nq={nq} is invalid. Must be a multiple of 3."
    gates_per_layer = nq // 3

    Lx_indices = np.random.randint(0, 144, (n_lin, gates_per_layer))
    Lb_indices = np.random.randint(0, 144, (n_lin, gates_per_layer))
    N_indices  = np.random.randint(0, 10752, (n_nonlin, gates_per_layer))

    beta = np.random.randint(0, 2, nq)
    if np.sum(beta) == 0: beta[0] = 1

    def apply_routing(state): return state.reshape((3, gates_per_layer), order='F').flatten()

    def apply_affine_block(x, indices_matrix):
        state = x.copy()
        for layer in range(len(indices_matrix)):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                M, c = DICT_144_LINEAR[indices_matrix[layer, g]]
                idx = slice(g * 3, g * 3 + 3)
                state[idx] = (M @ state[idx] + c) % 2
        return state

    zero_state = np.zeros(nq, dtype=int)
    b_vec = apply_affine_block(zero_state, Lb_indices)
    A = np.zeros((nq, nq), dtype=int)
    for i in range(nq):
        e_i = np.zeros(nq, dtype=int)
        e_i[i] = 1
        A[:, i] = apply_affine_block(e_i, Lb_indices) ^ b_vec

    beta_prime = (A.T @ beta) % 2

    learnable_parameters = (Lx_indices.flatten(), N_indices.flatten(), beta_prime.flatten())

    return {
        'learnable_params': learnable_parameters,
        'beta': beta,
        'b_vec': b_vec,
        'dict_144': DICT_144_LINEAR,
        'dict_10752': DICT_10752_NONLINEAR
    }

# --- 3. Standalone Forward Propagation (The Warm-up) ---

def evaluate_cmry_forward(learnable_params, nq, dict_144, dict_10752, X_batch=None):
    """
    Takes strictly the solver-learnable parameters and re-evaluates the Boolean function.
    Can be partially frozen using functools.partial(evaluate_cmry_forward, nq=..., dict_144=..., dict_10752=...)
    """
    Lx_idx_vec, N_idx_vec, beta_prime_vec = learnable_params

    # 1. Structural Assertions
    assert nq % 3 == 0, f"Topology Error: nq={nq} is not a multiple of 3."
    n_lin = math.ceil(math.log2(nq))
    n_nonlin = 6
    gates_per_layer = nq // 3

    expected_lx_len = n_lin * gates_per_layer
    expected_n_len = n_nonlin * gates_per_layer

    assert len(Lx_idx_vec) == expected_lx_len, f"Lx parameter mismatch. Expected {expected_lx_len}, got {len(Lx_idx_vec)}"
    assert len(N_idx_vec) == expected_n_len, f"N parameter mismatch. Expected {expected_n_len}, got {len(N_idx_vec)}"
    assert len(beta_prime_vec) == nq, f"beta' parameter mismatch. Expected {nq}, got {len(beta_prime_vec)}"

    # 2. Re-shape index vectors into layers for execution
    Lx_indices = Lx_idx_vec.reshape((n_lin, gates_per_layer))
    N_indices = N_idx_vec.reshape((n_nonlin, gates_per_layer))

    # 3. Define the layer execution logic
    def apply_routing(state):
        return state.reshape((3, gates_per_layer), order='F').flatten()

    def forward_pass_single(x):
        """Passes a single bitstring x through L_x and N."""
        state = x.copy()

        # Pass through L_x
        for layer in range(n_lin):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                M, c = dict_144[Lx_indices[layer, g]]
                idx = slice(g * 3, g * 3 + 3)
                state[idx] = (M @ state[idx] + c) % 2

        # Pass through N
        for layer in range(n_nonlin):
            state = apply_routing(state)
            for g in range(gates_per_layer):
                perm = dict_10752[N_indices[layer, g]]
                idx = slice(g * 3, g * 3 + 3)
                val = state[idx.start] + 2*state[idx.start+1] + 4*state[idx.start+2]
                out_val = perm[val]
                state[idx.start]   = (out_val >> 0) & 1
                state[idx.start+1] = (out_val >> 1) & 1
                state[idx.start+2] = (out_val >> 2) & 1

        return state

    # 4. Handle Batched vs Full Truth Table Evaluation
    if X_batch is None:
        if nq > 18:
            msg = (f"\n[WARNING] Truth table generation requested for nq={nq}. "
                   f"This requires {2**nq} evaluations and will block execution. "
                   f"Returning None. Provide X_batch to evaluate specific inputs.")
            warnings.warn(msg)
            return None

        # Generate the full hypercube of inputs
        num_states = 2**nq
        X_batch = np.array([[(i >> j) & 1 for j in range(nq)] for i in range(num_states)])

    # 5. Execute and compute final dot product with beta'
    F_batch = np.zeros(len(X_batch), dtype=int)
    for i, x in enumerate(X_batch):
        h = forward_pass_single(x)
        F_batch[i] = np.dot(beta_prime_vec, h) % 2

    return F_batch

# --- 4. Consistency Test ---

if __name__ == "__main__":
    NQ = 12
    print(f"Generating Ground Truth for nq={NQ}...")
    cmry_instance = generate_strict_cmry_instance(nq=NQ)

    learnable_params = cmry_instance['learnable_params']
    dict_144 = cmry_instance['dict_144']
    dict_10752 = cmry_instance['dict_10752']

    print("Testing Standalone Forward Propagator...")
    F_learned = evaluate_cmry_forward(
        learnable_params=learnable_params,
        nq=NQ,
        dict_144=dict_144,
        dict_10752=dict_10752,
        X_batch=None # Triggers full truth table since nq <= 18
    )

    print("Test Complete. F_learned generated successfully.")

Generating Ground Truth for nq=12...
Testing Standalone Forward Propagator...
Test Complete. F_learned generated successfully.


In [2]:
import jax
import jax.numpy as jnp
import numpy as np

# --- 1. Repacking Dictionaries into JAX PyTrees ---

def convert_dicts_to_pytrees(dict_144, dict_10752):
    """
    Converts Python lists of tuples/arrays into contiguous JAX arrays.
    """
    # dict_144 is a list of (M, c). We split it into two tensors.
    M_list = [item[0] for item in dict_144]
    c_list = [item[1] for item in dict_144]

    dict_144_M = jnp.array(M_list, dtype=jnp.uint8)     # Shape: (144, 3, 3)
    dict_144_c = jnp.array(c_list, dtype=jnp.uint8)     # Shape: (144, 3)

    dict_10752_tensor = jnp.array(dict_10752, dtype=jnp.uint8) # Shape: (10752, 8)

    return dict_144_M, dict_144_c, dict_10752_tensor

In [3]:
dict_144_M, dict_144_c, dict_10752_tensor = convert_dicts_to_pytrees(dict_144, dict_10752)

In [4]:
import jax
import jax.numpy as jnp
import numpy as np


# --- 2. The Vectorized JAX Forward Pass ---

@jax.jit
def compute_h_single_jax(x, Lx_indices, N_indices, dict_144_M, dict_144_c, dict_10752):
    """
    Computes h(x) = N(L_x(x)) for a single vector x.
    Inner gate loops are completely vectorized.
    """
    state = x
    n_lin = Lx_indices.shape[0]
    n_nonlin = N_indices.shape[0]
    gates_per_layer = Lx_indices.shape[1]

    # Pre-allocate the row indices for the advanced array lookup
    row_indices = jnp.arange(gates_per_layer)

    # 1. Forward through L_x
    for layer in range(n_lin):
        # Routing: Fortran-order reshape and flatten mimics the ternary stride
        state = jnp.reshape(state, (3, gates_per_layer), order='F').flatten()

        # Reshape to (G, 3) to process all gates in this layer simultaneously
        state_g = jnp.reshape(state, (gates_per_layer, 3))

        # Extract the matrices and shifts for this specific layer
        layer_gate_ids = Lx_indices[layer]
        M_layer = dict_144_M[layer_gate_ids]  # Shape: (G, 3, 3)
        c_layer = dict_144_c[layer_gate_ids]  # Shape: (G, 3)

        # Batched matrix-vector multiply over GF(2): vmap over the G dimension
        state_g = (jax.vmap(jnp.dot)(M_layer, state_g) + c_layer) % 2

        # Flatten back to 1D
        state = state_g.flatten()

    # 2. Forward through N
    for layer in range(n_nonlin):
        state = jnp.reshape(state, (3, gates_per_layer), order='F').flatten()
        state_g = jnp.reshape(state, (gates_per_layer, 3))

        layer_gate_ids = N_indices[layer]
        perms_layer = dict_10752[layer_gate_ids] # Shape: (G, 8)

        # Convert 3-bit arrays to integers (0-7)
        int_vals = state_g[:, 0] + 2 * state_g[:, 1] + 4 * state_g[:, 2]

        # Advanced indexing to lookup all G permutations simultaneously
        out_ints = perms_layer[row_indices, int_vals]

        # Bitwise extraction back to (G, 3)
        state_g = jnp.stack([
            out_ints & 1,
            (out_ints >> 1) & 1,
            (out_ints >> 2) & 1
        ], axis=-1)

        state = state_g.flatten()

    return state

@jax.jit
def batch_forward_propagate_jax(y_batch, v_batch, learnable_params_tuple, dict_144_M, dict_144_c, dict_10752):
    """
    Takes (K, nq) batches of y and upsilon, and returns the (K,) derivative vector b.
    """
    Lx_indices, N_indices, beta_prime = learnable_params_tuple

    # Vmap the single evaluator across the batch dimension (axis 0)
    # The dictionaries and parameter indices remain broadcasted (None)
    vmap_h = jax.vmap(compute_h_single_jax, in_axes=(0, None, None, None, None, None))

    # Compute h(y) and h(upsilon)
    h_y = vmap_h(y_batch, Lx_indices, N_indices, dict_144_M, dict_144_c, dict_10752)
    h_v = vmap_h(v_batch, Lx_indices, N_indices, dict_144_M, dict_144_c, dict_10752)

    # Delta h = h_y ^ h_v
    delta_h_batch = h_y ^ h_v

    # GF(2) dot product with beta_prime for the entire batch
    b_batch = jnp.dot(delta_h_batch, beta_prime) % 2

    return b_batch




Array([[[1, 1, 1],
        [1, 0, 1],
        [0, 1, 1]],

       [[1, 1, 1],
        [1, 0, 1],
        [0, 1, 1]],

       [[1, 1, 1],
        [1, 0, 1],
        [0, 1, 1]],

       ...,

       [[1, 0, 1],
        [1, 1, 0],
        [1, 1, 1]],

       [[1, 0, 1],
        [1, 1, 0],
        [1, 1, 1]],

       [[1, 0, 1],
        [1, 1, 0],
        [1, 1, 1]]], dtype=uint8)

In [6]:
import jax
import jax.numpy as jnp
import numpy as np
import math
import warnings

# --- 1. JIT-Compiled VMAP Core ---

@jax.jit
def _cmry_forward_core_jax(X_batch, Lx_indices, N_indices, beta_prime, dict_144_M, dict_144_c, dict_10752):
    """
    The strict XLA-compiled kernel for forward propagation.
    All python loops are unrolled based on the static shape of the index matrices.
    """

    def compute_single(x):
        state = x
        n_lin = Lx_indices.shape[0]
        n_nonlin = N_indices.shape[0]
        gates_per_layer = Lx_indices.shape[1]

        # Pre-allocate row indices for advanced indexing in the non-linear block
        row_indices = jnp.arange(gates_per_layer)

        # 1. Forward through L_x
        for layer in range(n_lin):
            state = jnp.reshape(state, (3, gates_per_layer), order='F').flatten()
            state_g = jnp.reshape(state, (gates_per_layer, 3))

            layer_gate_ids = Lx_indices[layer]
            M_layer = dict_144_M[layer_gate_ids]
            c_layer = dict_144_c[layer_gate_ids]

            # Batched GF(2) matrix-vector multiplication
            state_g = (jax.vmap(jnp.dot)(M_layer, state_g) + c_layer) % 2
            state = state_g.flatten()

        # 2. Forward through N
        for layer in range(n_nonlin):
            state = jnp.reshape(state, (3, gates_per_layer), order='F').flatten()
            state_g = jnp.reshape(state, (gates_per_layer, 3))

            layer_gate_ids = N_indices[layer]
            perms_layer = dict_10752[layer_gate_ids]

            # Binary to Integer
            int_vals = state_g[:, 0] + 2 * state_g[:, 1] + 4 * state_g[:, 2]

            # Advanced dictionary lookup
            out_ints = perms_layer[row_indices, int_vals]

            # Integer to Binary
            state_g = jnp.stack([
                out_ints & 1,
                (out_ints >> 1) & 1,
                (out_ints >> 2) & 1
            ], axis=-1)

            state = state_g.flatten()

        # 3. Final dot product with Beta'
        return jnp.dot(beta_prime, state) % 2

    # Map the single evaluator across the batch dimension (axis 0 of X_batch)
    return jax.vmap(compute_single)(X_batch)


# --- 2. Safe Python Wrapper ---

def evaluate_cmry_forward_jax(learnable_params, nq, dict_144_M, dict_144_c, dict_10752_tensor, X_batch=None):
    """
    Takes strictly the solver-learnable parameters and evaluates the Boolean function.
    Safely handles JAX data conversion and validates dimensions.
    """
    Lx_idx_vec, N_idx_vec, beta_prime_vec = learnable_params

    # 1. Structural Assertions
    assert nq % 3 == 0, f"Topology Error: nq={nq} is not a multiple of 3."
    n_lin = math.ceil(math.log2(nq))
    n_nonlin = 6
    gates_per_layer = nq // 3

    expected_lx_len = n_lin * gates_per_layer
    expected_n_len = n_nonlin * gates_per_layer

    assert len(Lx_idx_vec) == expected_lx_len, f"Lx parameter mismatch. Expected {expected_lx_len}, got {len(Lx_idx_vec)}"
    assert len(N_idx_vec) == expected_n_len, f"N parameter mismatch. Expected {expected_n_len}, got {len(N_idx_vec)}"
    assert len(beta_prime_vec) == nq, f"beta' parameter mismatch. Expected {nq}, got {len(beta_prime_vec)}"

    # 2. Handle Batched vs Full Truth Table Evaluation
    if X_batch is None:
        if nq > 18:
            msg = (f"\n[WARNING] Truth table generation requested for nq={nq}. "
                   f"This requires {2**nq} evaluations and will block execution. "
                   f"Returning None. Provide X_batch to evaluate specific inputs.")
            warnings.warn(msg)
            return None

        # Fast broadcasted NumPy generation of the hypercube
        num_states = 2**nq
        X_batch = ((np.arange(num_states, dtype=np.uint32)[:, None] >> np.arange(nq, dtype=np.uint32)) & 1).astype(np.uint8)

    # 3. Format Data for JAX execution
    Lx_indices = jnp.array(Lx_idx_vec, dtype=jnp.int32).reshape((n_lin, gates_per_layer))
    N_indices  = jnp.array(N_idx_vec, dtype=jnp.int32).reshape((n_nonlin, gates_per_layer))
    beta_prime = jnp.array(beta_prime_vec, dtype=jnp.uint8)
    X_batch_jax = jnp.array(X_batch, dtype=jnp.uint8)

    # 4. Bang. Execute the JIT Core
    F_batch = _cmry_forward_core_jax(
        X_batch_jax,
        Lx_indices,
        N_indices,
        beta_prime,
        dict_144_M,
        dict_144_c,
        dict_10752_tensor
    )

    return np.asarray(F_batch) # Convert back to standard NumPy array for external pipeline compatibility

In [7]:
F_learned

array([1, 0, 0, ..., 0, 1, 0])

In [9]:
import time

In [13]:
params = cmry_instance['learnable_params']
dict_144 = cmry_instance['dict_144']
dict_10752 = cmry_instance['dict_10752']

# 1. Convert dicts for JAX
M_list = [item[0] for item in dict_144]
c_list = [item[1] for item in dict_144]
dict_144_M = jnp.array(M_list, dtype=jnp.uint8)
dict_144_c = jnp.array(c_list, dtype=jnp.uint8)
dict_10752_tensor = jnp.array(dict_10752, dtype=jnp.uint8)

# 2. Run JAX (Includes compile time on first run)
print("Running JAX Evaluator (Compiling)...")
start = time.time()
F_jax = evaluate_cmry_forward_jax(
    params, NQ, dict_144_M, dict_144_c, dict_10752_tensor, X_batch=None
)
print(f"JAX Time (with compile): {time.time() - start:.4f} sec")

# 3. Run again (Compiled speed)
start = time.time()
F_jax = evaluate_cmry_forward_jax(
    params, NQ, dict_144_M, dict_144_c, dict_10752_tensor, X_batch=None
)
print(f"JAX Time (compiled): {time.time() - start:.4f} sec")

# 4. Compare to Numpy Ground Truth (without global phase shift)
# Note: cmry_instance['F_full'] includes the global phase.
# We XOR it with the global phase to isolate beta' dot h(x).
global_phase = np.dot(cmry_instance['beta'], cmry_instance['b_vec']) % 2
F_numpy = (F_learned ^ global_phase)

differences = np.sum(np.abs(F_jax - F_numpy))
print(f"\nDifferences between JAX and Numpy: {differences}")
assert differences == 0, "Mismatch detected!"
print("Verification Passed: Perfect Bijection Established.")

Running JAX Evaluator (Compiling)...
JAX Time (with compile): 0.0035 sec
JAX Time (compiled): 0.0032 sec

Differences between JAX and Numpy: 0
Verification Passed: Perfect Bijection Established.


In [2]:
cmry_12['learnable_params']

NameError: name 'cmry_12' is not defined

In [1]:
def extract_pure_gf2_parameters(Lx_indices, N_indices, beta_prime, dict_144, dict_10752):
    """
    Unpacks the physical ground truth into pure Boolean tensors.
    """
    n_lin, gates_per_layer = Lx_indices.shape
    n_nonlin = N_indices.shape[0]

    # 1. Unpack Linear Block into M (3x3) and c (3) bits
    Lx_M = np.zeros((n_lin, gates_per_layer, 3, 3), dtype=np.uint8)
    Lx_c = np.zeros((n_lin, gates_per_layer, 3), dtype=np.uint8)
    for l in range(n_lin):
        for g in range(gates_per_layer):
            Lx_M[l, g], Lx_c[l, g] = dict_144[Lx_indices[l, g]]

    # 2. Unpack Non-Linear Block into 8x3 Boolean Truth Tables
    N_T = np.zeros((n_nonlin, gates_per_layer, 8, 3), dtype=np.uint8)
    for l in range(n_nonlin):
        for g in range(gates_per_layer):
            perm = dict_10752[N_indices[l, g]] # This is an array of 8 integers
            # Convert the 8 integers into an 8x3 binary matrix
            binary_matrix = np.array([[(val >> 0) & 1, (val >> 1) & 1, (val >> 2) & 1] for val in perm])
            N_T[l, g] = binary_matrix

    return Lx_M, Lx_c, N_T, beta_prime.astype(np.uint8)

NumPy arrays are somewhat like native Python lists, except that

- Data *must be homogeneous* (all elements of the same type).  
- These types must be one of the [data types](https://docs.scipy.org/doc/numpy/reference/arrays.dtypes.html) (`dtypes`) provided by NumPy.  


The most important of these dtypes are:

- float64: 64 bit floating-point number  
- int64: 64 bit integer  
- bool:  8 bit True or False  


There are also dtypes to represent complex numbers, unsigned integers, etc.

On modern machines, the default dtype for arrays is `float64`

In [ ]:
a = np.zeros(3)
type(a[0])

If we want to use integers we can specify as follows:

In [ ]:
a = np.zeros(3, dtype=int)
type(a[0])


<a id='numpy-shape-dim'></a>

### Shape and Dimension


<a id='index-3'></a>
Consider the following assignment

In [ ]:
z = np.zeros(10)

Here `z` is a *flat* array with no dimension — neither row nor column vector.

The dimension is recorded in the `shape` attribute, which is a tuple

In [ ]:
z.shape

Here the shape tuple has only one element, which is the length of the array (tuples with one element end with a comma).

To give it dimension, we can change the `shape` attribute

In [ ]:
z.shape = (10, 1)
z

In [ ]:
z = np.zeros(4)
z.shape = (2, 2)
z

In the last case, to make the 2 by 2 array, we could also pass a tuple to the `zeros()` function, as
in `z = np.zeros((2, 2))`.


<a id='creating-arrays'></a>

### Creating Arrays


<a id='index-4'></a>
As we’ve seen, the `np.zeros` function creates an array of zeros.

You can probably guess what `np.ones` creates.

Related is `np.empty`, which creates arrays in memory that can later be populated with data

In [ ]:
z = np.empty(3)
z

The numbers you see here are garbage values.

(Python allocates 3 contiguous 64 bit pieces of memory, and the existing contents of those memory slots are interpreted as `float64` values)

To set up a grid of evenly spaced numbers use `np.linspace`

In [ ]:
z = np.linspace(2, 4, 5)  # From 2 to 4, with 5 elements

To create an identity matrix use either `np.identity` or `np.eye`

In [ ]:
z = np.identity(2)
z

In addition, NumPy arrays can be created from Python lists, tuples, etc. using `np.array`

In [ ]:
z = np.array([10, 20])                 # ndarray from Python list
z

In [ ]:
type(z)

In [ ]:
z = np.array((10, 20), dtype=float)    # Here 'float' is equivalent to 'np.float64'
z

In [ ]:
z = np.array([[1, 2], [3, 4]])         # 2D array from a list of lists
z

See also `np.asarray`, which performs a similar function, but does not make
a distinct copy of data already in a NumPy array.

In [ ]:
na = np.linspace(10, 20, 2)
na is np.asarray(na)   # Does not copy NumPy arrays

In [ ]:
na is np.array(na)     # Does make a new copy --- perhaps unnecessarily

To read in the array data from a text file containing numeric data use `np.loadtxt`
or `np.genfromtxt`—see [the documentation](http://docs.scipy.org/doc/numpy/reference/routines.io.html) for details.

### Array Indexing


<a id='index-5'></a>
For a flat array, indexing is the same as Python sequences:

In [ ]:
z = np.linspace(1, 2, 5)
z

In [ ]:
z[0]

In [ ]:
z[0:2]  # Two elements, starting at element 0

In [ ]:
z[-1]

For 2D arrays the index syntax is as follows:

In [ ]:
z = np.array([[1, 2], [3, 4]])
z

In [ ]:
z[0, 0]

In [ ]:
z[0, 1]

And so on.

Note that indices are still zero-based, to maintain compatibility with Python sequences.

Columns and rows can be extracted as follows

In [ ]:
z[0, :]

In [ ]:
z[:, 1]

NumPy arrays of integers can also be used to extract elements

In [ ]:
z = np.linspace(2, 4, 5)
z

In [ ]:
indices = np.array((0, 2, 3))
z[indices]

Finally, an array of `dtype bool` can be used to extract elements

In [ ]:
z

In [ ]:
d = np.array([0, 1, 1, 0, 0], dtype=bool)
d

In [ ]:
z[d]

We’ll see why this is useful below.

An aside: all elements of an array can be set equal to one number using slice notation

In [ ]:
z = np.empty(3)
z

In [ ]:
z[:] = 42
z

### Array Methods


<a id='index-6'></a>
Arrays have useful methods, all of which are carefully optimized

In [ ]:
a = np.array((4, 3, 2, 1))
a

In [ ]:
a.sort()              # Sorts a in place
a

In [ ]:
a.sum()               # Sum

In [ ]:
a.mean()              # Mean

In [ ]:
a.max()               # Max

In [ ]:
a.argmax()            # Returns the index of the maximal element

In [ ]:
a.cumsum()            # Cumulative sum of the elements of a

In [ ]:
a.cumprod()           # Cumulative product of the elements of a

In [ ]:
a.var()               # Variance

In [ ]:
a.std()               # Standard deviation

In [ ]:
a.shape = (2, 2)
a.T                   # Equivalent to a.transpose()

Another method worth knowing is `searchsorted()`.

If `z` is a nondecreasing array, then `z.searchsorted(a)` returns the index of the first element of `z` that is `>= a`

In [ ]:
z = np.linspace(2, 4, 5)
z

In [ ]:
z.searchsorted(2.2)

Many of the methods discussed above have equivalent functions in the NumPy namespace

In [ ]:
a = np.array((4, 3, 2, 1))

In [ ]:
np.sum(a)

In [ ]:
np.mean(a)

## Operations on Arrays


<a id='index-7'></a>

### Arithmetic Operations

The operators `+`, `-`, `*`, `/` and `**` all act *elementwise* on arrays

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([5, 6, 7, 8])
a + b

In [ ]:
a * b

We can add a scalar to each element as follows

In [ ]:
a + 10

Scalar multiplication is similar

In [ ]:
a * 10

The two-dimensional arrays follow the same general rules

In [ ]:
A = np.ones((2, 2))
B = np.ones((2, 2))
A + B

In [ ]:
A + 10

In [ ]:
A * B


<a id='numpy-matrix-multiplication'></a>
In particular, `A * B` is *not* the matrix product, it is an element-wise product.

### Matrix Multiplication


<a id='index-8'></a>
With Anaconda’s scientific Python package based around Python 3.5 and above,
one can use the `@` symbol for matrix multiplication, as follows:

In [ ]:
A = np.ones((2, 2))
B = np.ones((2, 2))
A @ B

(For older versions of Python and NumPy you need to use the [np.dot](http://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html) function)

We can also use `@` to take the inner product of two flat arrays

In [ ]:
A = np.array((1, 2))
B = np.array((10, 20))
A @ B

In fact, we can use `@` when one element is a Python list or tuple

In [ ]:
A = np.array(((1, 2), (3, 4)))
A

In [ ]:
A @ (0, 1)

Since we are post-multiplying, the tuple is treated as a column vector.

### Mutability and Copying Arrays

NumPy arrays are mutable data types, like Python lists.

In other words, their contents can be altered (mutated) in memory after initialization.

We already saw examples above.

Here’s another example:

In [ ]:
a = np.array([42, 44])
a

In [ ]:
a[-1] = 0  # Change last element to 0
a

Mutability leads to the following behavior (which can be shocking to MATLAB programmers…)

In [ ]:
a = np.random.randn(3)
a

In [ ]:
b = a
b[0] = 0.0
a

What’s happened is that we have changed `a` by changing `b`.

The name `b` is bound to `a` and becomes just another reference to the
array (the Python assignment model is described in more detail [later in the course](https://python.quantecon.org/python_advanced_features.html)).

Hence, it has equal rights to make changes to that array.

This is in fact the most sensible default behavior!

It means that we pass around only pointers to data, rather than making copies.

Making copies is expensive in terms of both speed and memory.

#### Making Copies

It is of course possible to make `b` an independent copy of `a` when required.

This can be done using `np.copy`

In [ ]:
a = np.random.randn(3)
a

In [ ]:
b = np.copy(a)
b

Now `b` is an independent copy (called a *deep copy*)

In [ ]:
b[:] = 1
b

In [ ]:
a

Note that the change to `b` has not affected `a`.

## Additional Functionality

Let’s look at some other useful things we can do with NumPy.

### Vectorized Functions


<a id='index-9'></a>
NumPy provides versions of the standard functions `log`, `exp`, `sin`, etc. that act *element-wise* on arrays

In [ ]:
z = np.array([1, 2, 3])
np.sin(z)

This eliminates the need for explicit element-by-element loops such as

In [ ]:
n = len(z)
y = np.empty(n)
for i in range(n):
    y[i] = np.sin(z[i])

Because they act element-wise on arrays, these functions are called *vectorized functions*.

In NumPy-speak, they are also called *ufuncs*, which stands for “universal functions”.

As we saw above, the usual arithmetic operations (`+`, `*`, etc.) also
work element-wise, and combining these with the ufuncs gives a very large set of fast element-wise functions.

In [ ]:
z

In [ ]:
(1 / np.sqrt(2 * np.pi)) * np.exp(- 0.5 * z**2)

Not all user-defined functions will act element-wise.

For example, passing the function `f` defined below a NumPy array causes a `ValueError`

In [ ]:
def f(x):
    return 1 if x > 0 else 0

The NumPy function `np.where` provides a vectorized alternative:

In [ ]:
x = np.random.randn(4)
x

In [ ]:
np.where(x > 0, 1, 0)  # Insert 1 if x > 0 true, otherwise 0

You can also use `np.vectorize` to vectorize a given function

In [ ]:
f = np.vectorize(f)
f(x)                # Passing the same vector x as in the previous example

However, this approach doesn’t always obtain the same speed as a more carefully crafted vectorized function.

### Comparisons


<a id='index-10'></a>
As a rule, comparisons on arrays are done element-wise

In [ ]:
z = np.array([2, 3])
y = np.array([2, 3])
z == y

In [ ]:
y[0] = 5
z == y

In [ ]:
z != y

The situation is similar for `>`, `<`, `>=` and `<=`.

We can also do comparisons against scalars

In [ ]:
z = np.linspace(0, 10, 5)
z

In [ ]:
z > 3

This is particularly useful for *conditional extraction*

In [ ]:
b = z > 3
b

In [ ]:
z[b]

Of course we can—and frequently do—perform this in one step

In [ ]:
z[z > 3]

### Sub-packages

NumPy provides some additional functionality related to scientific programming
through its sub-packages.

We’ve already seen how we can generate random variables using np.random

In [ ]:
z = np.random.randn(10000)  # Generate standard normals
y = np.random.binomial(10, 0.5, size=1000)    # 1,000 draws from Bin(10, 0.5)
y.mean()

Another commonly used subpackage is np.linalg

In [ ]:
A = np.array([[1, 2], [3, 4]])

np.linalg.det(A)           # Compute the determinant

In [ ]:
np.linalg.inv(A)           # Compute the inverse


<a id='index-12'></a>
Much of this functionality is also available in [SciPy](http://www.scipy.org/), a collection of modules that are built on top of NumPy.

We’ll cover the SciPy versions in more detail [soon](https://python.quantecon.org/scipy.html).

For a comprehensive list of what’s available in NumPy see [this documentation](https://docs.scipy.org/doc/numpy/reference/routines.html).

## Exercises


<a id='np-ex1'></a>

### Exercise 1

Consider the polynomial expression


<a id='equation-np-polynom'></a>
$$
p(x) = a_0 + a_1 x + a_2 x^2 + \cdots a_N x^N = \sum_{n=0}^N a_n x^n \tag{1}
$$

[Earlier](https://python.quantecon.org/python_essentials.html#pyess-ex2), you wrote a simple function `p(x, coeff)` to evaluate [(1)](#equation-np-polynom) without considering efficiency.

Now write a new function that does the same job, but uses NumPy arrays and array operations for its computations, rather than any form of Python loop.

(Such functionality is already implemented as `np.poly1d`, but for the sake of the exercise don’t use this class)

- Hint: Use `np.cumprod()`  



<a id='np-ex2'></a>

### Exercise 2

Let `q` be a NumPy array of length `n` with `q.sum() == 1`.

Suppose that `q` represents a [probability mass function](https://en.wikipedia.org/wiki/Probability_mass_function).

We wish to generate a discrete random variable $ x $ such that $ \mathbb P\{x = i\} = q_i $.

In other words, `x` takes values in `range(len(q))` and `x = i` with probability `q[i]`.

The standard (inverse transform) algorithm is as follows:

- Divide the unit interval $ [0, 1] $ into $ n $ subintervals $ I_0, I_1, \ldots, I_{n-1} $ such that the length of $ I_i $ is $ q_i $.  
- Draw a uniform random variable $ U $ on $ [0, 1] $ and return the $ i $ such that $ U \in I_i $.  


The probability of drawing $ i $ is the length of $ I_i $, which is equal to $ q_i $.

We can implement the algorithm as follows

In [ ]:
from random import uniform

def sample(q):
    a = 0.0
    U = uniform(0, 1)
    for i in range(len(q)):
        if a < U <= a + q[i]:
            return i
        a = a + q[i]

If you can’t see how this works, try thinking through the flow for a simple example, such as `q = [0.25, 0.75]`
It helps to sketch the intervals on paper.

Your exercise is to speed it up using NumPy, avoiding explicit loops

- Hint: Use `np.searchsorted` and `np.cumsum`  


If you can, implement the functionality as a class called `DiscreteRV`, where

- the data for an instance of the class is the vector of probabilities `q`  
- the class has a `draw()` method, which returns one draw according to the algorithm described above  


If you can, write the method so that `draw(k)` returns `k` draws from `q`.


<a id='np-ex3'></a>

### Exercise 3

Recall our [earlier discussion](https://python.quantecon.org/python_oop.html#oop-ex1) of the empirical cumulative distribution function.

Your task is to

1. Make the `__call__` method more efficient using NumPy.  
1. Add a method that plots the ECDF over $ [a, b] $, where $ a $ and $ b $ are method parameters.  

## Solutions

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

### Exercise 1

This code does the job

In [ ]:
def p(x, coef):
    X = np.ones_like(coef)
    X[1:] = x
    y = np.cumprod(X)   # y = [1, x, x**2,...]
    return coef @ y

Let’s test it

In [ ]:
x = 2
coef = np.linspace(2, 4, 3)
print(coef)
print(p(x, coef))
# For comparison
q = np.poly1d(np.flip(coef))
print(q(x))

### Exercise 2

Here’s our first pass at a solution:

In [ ]:
from numpy import cumsum
from numpy.random import uniform

class DiscreteRV:
    """
    Generates an array of draws from a discrete random variable with vector of
    probabilities given by q.
    """

    def __init__(self, q):
        """
        The argument q is a NumPy array, or array like, nonnegative and sums
        to 1
        """
        self.q = q
        self.Q = cumsum(q)

    def draw(self, k=1):
        """
        Returns k draws from q. For each such draw, the value i is returned
        with probability q[i].
        """
        return self.Q.searchsorted(uniform(0, 1, size=k))

The logic is not obvious, but if you take your time and read it slowly,
you will understand.

There is a problem here, however.

Suppose that `q` is altered after an instance of `discreteRV` is
created, for example by

In [ ]:
q = (0.1, 0.9)
d = DiscreteRV(q)
d.q = (0.5, 0.5)

The problem is that `Q` does not change accordingly, and `Q` is the
data used in the `draw` method.

To deal with this, one option is to compute `Q` every time the draw
method is called.

But this is inefficient relative to computing `Q` once-off.

A better option is to use descriptors.

A solution from the [quantecon
library](https://github.com/QuantEcon/QuantEcon.py/tree/master/quantecon)
using descriptors that behaves as we desire can be found
[here](https://github.com/QuantEcon/QuantEcon.py/blob/master/quantecon/discrete_rv.py).

### Exercise 3

An example solution is given below.

In essence, we’ve just taken [this
code](https://github.com/QuantEcon/QuantEcon.py/blob/master/quantecon/ecdf.py)
from QuantEcon and added in a plot method

In [ ]:
"""
Modifies ecdf.py from QuantEcon to add in a plot method

"""

class ECDF:
    """
    One-dimensional empirical distribution function given a vector of
    observations.

    Parameters
    ----------
    observations : array_like
        An array of observations

    Attributes
    ----------
    observations : array_like
        An array of observations

    """

    def __init__(self, observations):
        self.observations = np.asarray(observations)

    def __call__(self, x):
        """
        Evaluates the ecdf at x

        Parameters
        ----------
        x : scalar(float)
            The x at which the ecdf is evaluated

        Returns
        -------
        scalar(float)
            Fraction of the sample less than x

        """
        return np.mean(self.observations <= x)

    def plot(self, ax, a=None, b=None):
        """
        Plot the ecdf on the interval [a, b].

        Parameters
        ----------
        a : scalar(float), optional(default=None)
            Lower endpoint of the plot interval
        b : scalar(float), optional(default=None)
            Upper endpoint of the plot interval

        """

        # === choose reasonable interval if [a, b] not specified === #
        if a is None:
            a = self.observations.min() - self.observations.std()
        if b is None:
            b = self.observations.max() + self.observations.std()

        # === generate plot === #
        x_vals = np.linspace(a, b, num=100)
        f = np.vectorize(self.__call__)
        ax.plot(x_vals, f(x_vals))
        plt.show()

Here’s an example of usage

In [ ]:
fig, ax = plt.subplots()
X = np.random.randn(1000)
F = ECDF(X)
F.plot(ax)